# WasteVision AI — Model Training in Google Colab (Free GPU / ₹0)

This notebook trains YOLOv8 / YOLO11 waste detection models using the dataset exported from your **WasteVision AI** platform directly on Google Drive.

**Architecture:**
- **Next.js Web App**: Team labelling interface, bounding box drawer, review workflow.
- **Supabase PostgreSQL**: Metadata, classes, user roles, assignments.
- **Google Drive**: Original images, YOLO `.txt` labels, and exported datasets.
- **Google Colab (this notebook)**: Free T4 GPU training for ₹0 cost.

## Step 1: Check GPU Acceleration
Verify that Google Colab is using a GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU active: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Go to Runtime → Change runtime type and select T4 GPU.")

## Step 2: Mount Google Drive
Mount your Google Drive to access the dataset exported by WasteVision AI.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

## Step 3: Install Ultralytics (YOLOv8 / YOLO11)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## Step 4: Configure Dataset Path
Set the path to your exported WasteVision dataset folder on Google Drive.
WasteVision AI exports datasets to your Drive's `exports/` folder with `dataset.yaml` and YOLO format labels.

In [ ]:
import os
from pathlib import Path

# Update this path to match your exported dataset folder in Drive:
# Example: /content/drive/MyDrive/WasteVision/exports/WasteVision_waste_detector_v1
DATASET_DIR = "/content/drive/MyDrive/WasteVision/exports/WasteVision_waste_detector_v1"

yaml_path = Path(DATASET_DIR) / "dataset.yaml"
if yaml_path.exists():
    print(f"✅ Found dataset configuration at: {yaml_path}")
    print("\n--- dataset.yaml preview ---"
          )
    print(yaml_path.read_text())
else:
    print(f"❌ dataset.yaml not found at {yaml_path}.")
    print("Please verify your Drive folder path above or run 'Export YOLO to Drive' on the WasteVision website.")

## Step 5: Train YOLOv8 Waste Detector
Select your preferred model architecture:
- `yolov8n.pt`: Nano model — ultra-fast, small footprint.
- `yolov8s.pt`: Small model — higher accuracy on smaller waste objects (~2 cm).

In [ ]:
from ultralytics import YOLO

# Choose model: yolov8n.pt (Nano) or yolov8s.pt (Small)
MODEL_NAME = "yolov8n.pt"
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16

model = YOLO(MODEL_NAME)

# Train on GPU
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    project="/content/drive/MyDrive/WasteVision/runs",
    name="waste_sorter_run",
    exist_ok=True
)
print("\n🎉 Training Complete!")

## Step 6: Validate and View Metrics

In [ ]:
# Validate the best checkpoint on the validation split
metrics = model.val()
print(f"Mean Average Precision (mAP50): {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## Step 7: Display Training Results and Confusion Matrix

In [ ]:
from IPython.display import Image, display

run_dir = Path("/content/drive/MyDrive/WasteVision/runs/waste_sorter_run")

# Show confusion matrix
cm = run_dir / "confusion_matrix.png"
if cm.exists():
    print("Confusion Matrix:")
    display(Image(filename=str(cm)))

# Show training results curves
res_img = run_dir / "results.png"
if res_img.exists():
    print("\nTraining Loss & Metrics per Epoch:")
    display(Image(filename=str(res_img)))

# Show sample validation predictions
val_pred = run_dir / "val_batch0_pred.jpg"
if val_pred.exists():
    print("\nSample Validation Predictions:")
    display(Image(filename=str(val_pred)))

## Step 8: Save Trained Weights
Your best model weights are automatically saved to your Drive at:
`/content/drive/MyDrive/WasteVision/runs/waste_sorter_run/weights/best.pt`

You can deploy this model directly to your physical waste-sorter camera hardware (Raspberry Pi, Jetson Nano, or edge PC)!